# Building a multiplayer game server
* lesson 33 gave us the raw building blocks: sockets, TCP/UDP, message framing, and the authoritative-server idea
* this lesson builds a real, working multiplayer server: multiple players connect at once, move around a shared little world, and each player's screen stays in sync with everyone else's - the actual core of "a multiplayer game"
* we use `asyncio` (recap lesson 11) instead of raw threads for this - handling MANY client connections at once is exactly the "lots of I/O-bound waiting" scenario asyncio is built for, and unlike lesson 11's threading race-condition example, a single-threaded asyncio server never has two pieces of code touching shared state at the EXACT same instant, so we dont need any locks here at all
* jupyter notebook reminder (recap lesson 11): we use top-level `await` directly in cells instead of `asyncio.run()`, since the notebook's own kernel already has an event loop running

In [1]:
import asyncio
import json

# --- the server's AUTHORITATIVE state - only code running on the "server side" below is ever allowed to change this ---
player_state = {}    # player_id -> {"x": int, "y": int} - the real, true position of every connected player
connections = {}     # player_id -> the asyncio StreamWriter used to send THAT player messages
next_player_number = [1]  # a mutable single-element list so our nested function below can bump it (recap "nonlocal"-ish tricks, lesson 04)

# the server's own copy of the world map - clients never get to modify this, only ASK the server to move them on it
WORLD_MAP = [
    "########",
    "#......#",
    "#..##..#",
    "#..##..#",
    "#......#",
    "########",
]


def is_walkable(x, y):
    """Authoritative collision check (recap lesson 31) - the server is the ONLY code allowed to decide this."""
    if not (0 <= y < len(WORLD_MAP)) or not (0 <= x < len(WORLD_MAP[0])):
        return False
    return WORLD_MAP[y][x] != "#"


async def broadcast_state():
    """Send the current, real player_state to EVERY connected client - this is how everyone stays in sync."""
    message = (json.dumps({"type": "state", "players": dict(player_state)}) + "\n").encode()
    for writer in connections.values():
        writer.write(message)
        await writer.drain()  # wait until the OS has actually accepted the bytes for sending, recap asyncio streams


async def handle_client(reader, writer):
    """This coroutine runs once PER connected client, for as long as that client stays connected."""
    player_id = f"player{next_player_number[0]}"
    next_player_number[0] += 1
    player_state[player_id] = {"x": 1, "y": 1}  # everyone starts in the same top-left open corner
    connections[player_id] = writer

    writer.write((json.dumps({"type": "welcome", "player_id": player_id}) + "\n").encode())
    await writer.drain()
    await broadcast_state()  # let everyone (including the new player) see the updated player list right away

    try:
        while True:
            line = await reader.readline()  # asyncio streams give us readline() for free - no manual buffering (recap lesson 33!)
            if not line:  # an empty read means the client closed the connection
                break
            request = json.loads(line)
            if request["type"] == "move":
                # THE authoritative check - exactly the lesson 33 pattern, just now inside a real multi-client server
                current = player_state[player_id]
                new_x, new_y = current["x"] + request["dx"], current["y"] + request["dy"]
                if is_walkable(new_x, new_y):
                    current["x"], current["y"] = new_x, new_y
                await broadcast_state()  # tell EVERYONE about the (possibly unchanged, if the move was rejected) new state
    finally:
        # this runs whether the client disconnected cleanly or the connection just died - always clean up after ourselves
        del player_state[player_id]
        del connections[player_id]
        await broadcast_state()  # let the remaining players see that this player is gone


game_server = await asyncio.start_server(handle_client, "127.0.0.1", 0)
game_port = game_server.sockets[0].getsockname()[1]
print(f"multiplayer game server listening on 127.0.0.1:{game_port}")

multiplayer game server listening on 127.0.0.1:38183


## a little `SimClient` helper - what a real game client's network layer looks like
* a real multiplayer game client does two things constantly, AT THE SAME TIME: it listens for incoming updates from the server (so the screen stays current, even from other players' actions), and it sends the LOCAL player's own actions whenever they happen
* `asyncio.create_task()` (recap lesson 11) is exactly the tool for "start this and let it keep running in the background while I do other things" - we launch one background "listener" task per client, so it keeps receiving broadcasts even while our demo code below is busy doing something else

In [2]:
class SimClient:
    """Stands in for "one player's game client" - connects, listens for broadcasts in the background, sends moves."""

    def __init__(self, name):
        self.name = name
        self.history = []  # every state update this client has ever received, in order - our "what did this player see"

    async def connect(self):
        self.reader, self.writer = await asyncio.open_connection("127.0.0.1", game_port)
        welcome = json.loads(await self.reader.readline())
        self.player_id = welcome["player_id"]
        self._listen_task = asyncio.create_task(self._listen_forever())  # background task, keeps running independently

    async def _listen_forever(self):
        """Runs in the background for this client's whole lifetime, recording every broadcast as it arrives."""
        while True:
            line = await self.reader.readline()
            if not line:
                break
            self.history.append(json.loads(line))

    async def move(self, dx, dy):
        """Send a move REQUEST - recap lesson 33/31: this is only ever a request, the server decides what really happens."""
        self.writer.write((json.dumps({"type": "move", "dx": dx, "dy": dy}) + "\n").encode())
        await self.writer.drain()

    async def disconnect(self):
        self.writer.close()
        await asyncio.sleep(0.05)  # give the server a moment to notice and process the disconnect

    @property
    def latest_state(self):
        """The most recent "state" broadcast this client saw - skips over other message types like "tick"/"chat"."""
        for entry in reversed(self.history):
            if entry["type"] == "state":
                return entry["players"]
        return {}


print("SimClient ready - it connects, listens in the background, and can send move requests")

SimClient ready - it connects, listens in the background, and can send move requests


## two players, moving around, staying in sync
* now the real demo: two SEPARATE clients connect, move around independently, and we prove that BOTH of them end up seeing the exact same, correct final state - exactly what "staying in sync in a multiplayer game" means in practice
* we sprinkle in small `asyncio.sleep()` calls between actions purely so the background listener tasks have a moment to actually receive and record each broadcast before we check `.history` - a real game wouldnt need this, it would just keep rendering whatever the latest received state is, whenever it arrives

In [3]:
alice = SimClient("Alice")
bob = SimClient("Bob")

await alice.connect()
await bob.connect()
await asyncio.sleep(0.1)  # let both "welcome" + initial broadcasts settle

print(f"{alice.name} connected as {alice.player_id}, {bob.name} connected as {bob.player_id}")
print("state right after both joined:", alice.latest_state)

await alice.move(1, 0)   # Alice steps right, from (1,1) to (2,1) - a walkable floor tile
await asyncio.sleep(0.1)
await bob.move(0, 1)     # Bob steps down, from (1,1) to (1,2) - also walkable
await asyncio.sleep(0.1)

print()
print(f"{alice.name}'s view of the world:", alice.latest_state)
print(f"{bob.name}'s view of the world:  ", bob.latest_state)
print("do both players agree on the current state?", alice.latest_state == bob.latest_state)

Alice connected as player1, Bob connected as player2
state right after both joined: {'player1': {'x': 1, 'y': 1}, 'player2': {'x': 1, 'y': 1}}



Alice's view of the world: {'player1': {'x': 2, 'y': 1}, 'player2': {'x': 1, 'y': 2}}
Bob's view of the world:   {'player1': {'x': 2, 'y': 1}, 'player2': {'x': 1, 'y': 2}}
do both players agree on the current state? True


## client-side prediction and server reconciliation
* recap lesson 33's latency point: waiting for a server round-trip before showing ANY movement would feel sluggish in a fast game
* **client-side prediction**: the client immediately updates its OWN local guess of its position the instant the player presses a key, without waiting for the server's answer at all - it just assumes the move will probably be legal
* **server reconciliation**: when the server's real, authoritative answer eventually arrives, the client compares it to its own guess - if they match, great, nothing visible happens; if they DISAGREE (the move was actually illegal, or something else changed), the client snaps/corrects itself to match the server's real answer
* lets simulate exactly this: Alice predicts a move that (unknown to her client) the server is guaranteed to reject - stepping WAY off the edge of the map - displays it immediately, then gets corrected once the real server answer comes back

In [4]:
# a move guaranteed to be illegal no matter where alice currently stands: 10 tiles up is always off the map entirely
predicted_position = dict(alice.latest_state[alice.player_id])  # a LOCAL copy - "what alice's client THINKS is true, right now"
print("alice's client-side position before predicting:", predicted_position)

# client-side prediction: update the LOCAL copy immediately, before the server has replied at all
attempted_dx, attempted_dy = 0, -10  # deliberately illegal - this always lands outside the map, guaranteed rejected
predicted_position["x"] += attempted_dx
predicted_position["y"] += attempted_dy
print("alice's client PREDICTS (shows immediately, optimistically):", predicted_position)

# now actually send the request and wait for the real, authoritative answer
await alice.move(attempted_dx, attempted_dy)
await asyncio.sleep(0.1)
authoritative_position = alice.latest_state[alice.player_id]
print("the SERVER's real, authoritative answer arrives:          ", authoritative_position)

if predicted_position == authoritative_position:
    print("prediction was correct - nothing visible needs to change, the player never even noticed a delay")
else:
    print("prediction was WRONG - a real client would now snap back / correct itself to the server's real answer")
    print("this is server reconciliation: the server's version always wins, no matter what the client guessed")

alice's client-side position before predicting: {'x': 2, 'y': 1}
alice's client PREDICTS (shows immediately, optimistically): {'x': 2, 'y': -9}
the SERVER's real, authoritative answer arrives:           {'x': 2, 'y': 1}
prediction was WRONG - a real client would now snap back / correct itself to the server's real answer
this is server reconciliation: the server's version always wins, no matter what the client guessed


## a server "tick rate" - broadcasting on a schedule, not just on request
* so far our server only broadcasts when a player sends a move - fine for a turn-based-feeling demo, but many real-time games ALSO run a fixed-rate server loop (a "tick"), e.g. 20 or 60 times per second, broadcasting the current state on a schedule regardless of whether anything player-triggered happened
* why bother, if nothing changed? real games often have server-side things happening WITHOUT player input at all (an enemy moving on its own, a timer counting down, physics simulating) - a tick loop is what keeps broadcasting those changes too
* we simulate a tiny tick loop here: 5 ticks, a fixed small delay between each, incrementing a simple server-side "world_time" counter and broadcasting it every tick - a fixed number of ticks so this cell actually finishes, instead of running forever like a real server's tick loop would

In [5]:
world_time = {"tick": 0}

async def broadcast_tick():
    message = (json.dumps({"type": "tick", "tick": world_time["tick"]}) + "\n").encode()
    for writer in connections.values():
        writer.write(message)
        await writer.drain()

TICK_COUNT = 5
TICK_DELAY = 0.05  # seconds between ticks - a real game might use something like 1/60 for a "60 tick" server

for _ in range(TICK_COUNT):
    world_time["tick"] += 1
    await broadcast_tick()
    await asyncio.sleep(TICK_DELAY)

await asyncio.sleep(0.1)  # let the listener tasks catch up on the last couple of broadcasts
print(f"{alice.name} received {sum(1 for entry in alice.history if entry['type'] == 'tick')} tick broadcasts")
print("last tick alice saw:", [entry for entry in alice.history if entry["type"] == "tick"][-1])

Alice received 5 tick broadcasts
last tick alice saw: {'type': 'tick', 'tick': 5}


## handling a disconnect
* a player's connection can end at any moment - closing the app normally, losing internet, crashing, anything
* our server's `finally:` block (recap lesson 06's error handling - `finally` always runs) already handles this: whenever `handle_client`'s loop ends, for ANY reason, we remove that player from `player_state`/`connections` and broadcast the update, so everyone else's game correctly shows that player as gone

In [6]:
print("before bob disconnects:", alice.latest_state)

await bob.disconnect()
await asyncio.sleep(0.1)  # give the server a moment to notice the closed connection and broadcast the update

print("after bob disconnects: ", alice.latest_state)
print("bob's player_id still present?", bob.player_id in alice.latest_state)

before bob disconnects: {'player1': {'x': 2, 'y': 1}, 'player2': {'x': 1, 'y': 2}}
after bob disconnects:  {'player1': {'x': 2, 'y': 1}}
bob's player_id still present? False


## cleanup
* time to close everything down properly, same habit as every networked lesson in this course

In [7]:
await alice.disconnect()
game_server.close()
await game_server.wait_closed()
print("multiplayer server closed")

multiplayer server closed


## quick comparison to other languages/tools
* python's `asyncio` handling many concurrent client connections on a single thread is architecturally very close to **Node.js** - Node's entire networking story is built around exactly this "single-threaded event loop, lots of concurrent I/O" model (recap lesson 11's comparison), which is a big part of why Node.js is such a popular choice for real-time multiplayer backends (e.g. the `Colyseus` framework, built specifically for multiplayer game servers)
* Go takes a different but equally popular approach for this: goroutines + channels (recap lesson 11) make handling thousands of concurrent connections with real OS-level concurrency almost as easy to write as our asyncio version, without needing an event loop at all
* our "broadcast the full state to everyone, every time something changes" approach is simple and great for learning, but doesn't scale forever - real, bigger multiplayer games often send DELTA updates (only what changed since last time, not the whole world) to save bandwidth, and use interest management ("only tell a client about players/objects actually near them") for very large worlds
* dedicated multiplayer-game backend services exist specifically so studios don't have to write all of this from scratch: **Photon**, **Colyseus**, **Nakama**, and (built into their respective engines) Unity's Netcode and Godot's high-level multiplayer API all provide a ready-made version of the state-sync + authoritative-server pattern we just built by hand

## Exercises
1. add a `"chat"` message type: a client sends `{"type": "chat", "text": "..."}`, the server broadcasts it to everyone as `{"type": "chat", "player_id": ..., "text": ...}` (recap lesson 33's point about TCP being the right choice for chat, not UDP)
2. connect a THIRD `SimClient` to the running server pattern (rebuild the server fresh for this exercise, since we closed the original one above) and confirm all three clients see each other in the broadcast state
3. simulate a REJECTED move (deliberately move a client into a wall tile) and confirm, by checking `.latest_state`, that the position did NOT actually change on the server
4. add a simple "last move timestamp" per player (`time.time()` when they last successfully moved) to `player_state`, broadcast it too, and print how long ago each player last moved

In [8]:
# TODO: solve exercise 1 here (add a chat message type, broadcast it to everyone)


# TODO: solve exercise 2 here (a fresh server + 3 SimClients, confirm all 3 appear in each other's state)


# TODO: solve exercise 3 here (deliberately move into a wall, confirm the server rejected it)


# TODO: solve exercise 4 here (track + broadcast a last-move timestamp per player)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [9]:
import time

# --- a fresh server for the exercises, since we closed the original one above ---
ex_player_state = {}
ex_connections = {}
ex_next_id = [1]
ex_last_move_time = {}

async def ex_broadcast():
    message = (json.dumps({
        "type": "state",
        "players": dict(ex_player_state),
        "last_move": dict(ex_last_move_time),
    }) + "\n").encode()
    for w in ex_connections.values():
        w.write(message)
        await w.drain()

async def ex_handle_client(reader, writer):
    player_id = f"player{ex_next_id[0]}"; ex_next_id[0] += 1
    ex_player_state[player_id] = {"x": 1, "y": 1}
    ex_connections[player_id] = writer
    writer.write((json.dumps({"type": "welcome", "player_id": player_id}) + "\n").encode())
    await writer.drain()
    await ex_broadcast()
    try:
        while True:
            line = await reader.readline()
            if not line:
                break
            request = json.loads(line)
            if request["type"] == "move":
                # sample solution 3 - the wall-collision check already rejects illegal moves, we just verify it below
                current = ex_player_state[player_id]
                new_x, new_y = current["x"] + request["dx"], current["y"] + request["dy"]
                if is_walkable(new_x, new_y):
                    current["x"], current["y"] = new_x, new_y
                    ex_last_move_time[player_id] = time.time()  # sample solution 4 - record when a move actually succeeded
                await ex_broadcast()
            elif request["type"] == "chat":  # sample solution 1 - a new message type, just broadcast it to everyone
                chat_message = (json.dumps({"type": "chat", "player_id": player_id, "text": request["text"]}) + "\n").encode()
                for w in ex_connections.values():
                    w.write(chat_message)
                    await w.drain()
    finally:
        del ex_player_state[player_id]
        del ex_connections[player_id]
        ex_last_move_time.pop(player_id, None)
        await ex_broadcast()

exercise_server = await asyncio.start_server(ex_handle_client, "127.0.0.1", 0)
exercise_port = exercise_server.sockets[0].getsockname()[1]

class ExSimClient(SimClient):
    async def connect(self):
        self.reader, self.writer = await asyncio.open_connection("127.0.0.1", exercise_port)
        welcome = json.loads(await self.reader.readline())
        self.player_id = welcome["player_id"]
        self._listen_task = asyncio.create_task(self._listen_forever())
    async def send_chat(self, text):
        self.writer.write((json.dumps({"type": "chat", "text": text}) + "\n").encode())
        await self.writer.drain()

# sample solution 2 - three clients on the same fresh server
p1, p2, p3 = ExSimClient("P1"), ExSimClient("P2"), ExSimClient("P3")
await p1.connect(); await p2.connect(); await p3.connect()
await asyncio.sleep(0.1)
print("all three players visible to p1?", set(p1.latest_state) == {p1.player_id, p2.player_id, p3.player_id})

# sample solution 1 - send a chat message and confirm everyone receives it
await p1.send_chat("hello everyone!")
await asyncio.sleep(0.1)
p2_chats = [entry for entry in p2.history if entry["type"] == "chat"]
print("p2 received chat:", p2_chats[-1])

# sample solution 3 - deliberately walk p1 into the inner wall block and confirm it gets rejected
# a move 10 tiles to the right is always off the map, no matter which map or starting tile - guaranteed illegal
before = dict(p1.latest_state[p1.player_id])
await p1.move(10, 0)
await asyncio.sleep(0.1)
after = p1.latest_state[p1.player_id]
print("position before illegal move:", before, "| after (should be unchanged):", after, "| rejected correctly?", before == after)

# sample solution 4 - make one legal move, then check its last-move timestamp got recorded
await p2.move(1, 0)  # a normal, legal step - this one SHOULD update ex_last_move_time
await asyncio.sleep(0.1)
print("recorded last-move timestamps:", p2.history[-1].get("last_move"))

await p1.disconnect(); await p2.disconnect(); await p3.disconnect()
exercise_server.close()
await exercise_server.wait_closed()
print("exercise server closed")

all three players visible to p1? True


p2 received chat: {'type': 'chat', 'player_id': 'player1', 'text': 'hello everyone!'}


position before illegal move: {'x': 1, 'y': 1} | after (should be unchanged): {'x': 1, 'y': 1} | rejected correctly? True


recorded last-move timestamps: {'player2': 1790421690.654105}


exercise server closed


* congratulation you finished this lesson

## what we learned
* building a real multi-client server with `asyncio.start_server()`, one `handle_client()` coroutine running per connection
* why asyncio's single-threaded model avoids the race conditions lesson 11 showed us with real threads, with no locks needed
* the broadcast pattern: whenever the authoritative state changes, tell every connected client, so everyone stays in sync
* a `SimClient` helper with a background listener task (`asyncio.create_task`) - exactly the shape a real game client's networking code takes: keep listening AND keep sending, at the same time
* client-side prediction (show the move immediately, optimistically) and server reconciliation (correct the client if the server's real answer ever disagrees)
* a fixed-rate server "tick" loop, broadcasting on a schedule instead of only in response to player input
* handling disconnects cleanly with `finally`, so the rest of the game correctly reflects a player leaving
* how this all maps onto real multiplayer tools and architectures (Node.js/Colyseus, Go, Photon, Nakama, delta updates and interest management for bigger games)